# Property label evaluation

Input (dataset V2): https://github.com/adrian9631/ravenclaw_project/tree/main/data_collection/data/dataset_v2/entities_data)

This notebook evaluates the output from https://docs.google.com/document/d/1KQGITNrTLobLbaZG9bvrM7hnIkwOWARpOyQRUmgG4nY/

1. For how many entities from dataset V2 do we have answers in all target languages for all properties ("complete entities")? 
2. How many labels are missing for each property? 

In [15]:
from pathlib import Path
import json
import polars as pl

In [16]:
input_dir = Path("data")
input_files = {file.stem: file for file in input_dir.glob("*.json")}

In [19]:
for language in input_files.keys():
    print(language)
    entities_wide = pl.read_json(input_files[language])
    # Reshape df and drop unnecessary columns
    entities_long = entities_wide.unpivot().unnest("value").drop(["labels", "labels_count", "sitelinks"])
    print(f"Number of entities: {entities_long.shape[0]}")
    # Extract label counts for each property
    entities_labels = (
        entities_long
            .select(["variable", pl.col("^P.*$")])     
            .with_columns(
                pl.col("^P.*$")
                    .struct.field("values")                                  
                    .list.eval(pl.element().struct.field("labels_count"))                                     
                    .name.keep()                                            
            )
    )
    #print(entities_labels)
    entities_bools = (
        entities_labels
            .select(pl.col("^P.*$")
                .list.contains(9)
                .fill_null(False)
            )
    )
    
    # Count how many entities have at least one label with label_count of 9, 
    # and how many entities have at least one label with label_count of 9 for ALL properties 
    # all_horizontal performs a row-wise AND across relevant properties
    label_counts = (
        entities_bools
            .select(
                pl.col("^P.*$").sum().cast(pl.Int64).name.keep(),
                pl.all_horizontal(pl.exclude(["P570", "P569"]))
                    .sum()
                    .cast(pl.Int64)
                    .alias("complete")
            )
    )
       
    with pl.Config(
        tbl_formatting="MARKDOWN",
        tbl_hide_column_data_types=True,
        tbl_hide_dataframe_shape=True,
    ):
        print(label_counts)

Arabic
Number of entities: 853
| P570 | P20 | P19 | P569 | P106 | P27 | complete |
|------|-----|-----|------|------|-----|----------|
| 0    | 720 | 607 | 0    | 853  | 816 | 514      |
Chinese
Number of entities: 632
| P570 | P20 | P19 | P569 | P106 | P27 | complete |
|------|-----|-----|------|------|-----|----------|
| 0    | 509 | 326 | 0    | 632  | 618 | 267      |
French
Number of entities: 4747
| P570 | P20  | P19  | P569 | P106 | P27  | complete |
|------|------|------|------|------|------|----------|
| 0    | 2756 | 2390 | 0    | 4746 | 4577 | 1503     |
Polish
Number of entities: 847
| P570 | P20 | P19 | P569 | P106 | P27 | complete |
|------|-----|-----|------|------|-----|----------|
| 0    | 647 | 444 | 0    | 847  | 755 | 345      |
German
Number of entities: 4577
| P570 | P20  | P19  | P569 | P106 | P27  | complete |
|------|------|------|------|------|------|----------|
| 0    | 2928 | 2395 | 0    | 4577 | 4257 | 1549     |
English
Number of entities: 14376
| P570 | P

In [82]:
# check output
entities_long.select(pl.col("^P.*$")).unnest("P106")
entities_long.select(pl.col("^P.*$")).unnest("P570")
# P570 and P569 are dates, they don't have labels
# P106 are occupations, some entities have multiple occupations

values,count,P20,P19,P569,P106,P27
list[struct[3]],i64,struct[2],struct[2],struct[2],struct[2],struct[2]
"[{""2001-11-03T00:00:00Z"",{},0}]",1,"{[{""http://www.wikidata.org/entity/Q157686"",{""Campiglia Marittima"",""Campiglia Marittima"",""Campiglia Marittima"",""Кампилья-Мариттима"",null,""滨海坎皮利亚"",""Campiglia Marittima"",""Campiglia Marittima"",""كامبيجليا ماريتيما""},8}],1}","{[{""http://www.wikidata.org/entity/Q220"",{""Rome"",""Rom"",""Rome"",""Рим"",""रोम"",""罗马"",""Roma"",""Rzym"",""روما""},9}],1}","{[{""1924-12-08T00:00:00Z"",{},0}],1}","{[{""http://www.wikidata.org/entity/Q4964182"",{""philosopher"",""Philosoph"",""philosophe"",""философ"",""दार्शनिक"",""哲學家"",""filosofo"",""filozof"",""فيلسوف""},9}, {""http://www.wikidata.org/entity/Q82955"",{""politician"",""Politiker"",""personnalité politique"",""политик"",""राजनीतिज्ञ"",""政治人物"",""politico"",""polityk"",""سياسي""},9}],2}","{[{""http://www.wikidata.org/entity/Q172579"",{""Kingdom of Italy"",""Königreich Italien"",""Royaume d'Italie"",""Королевство Италия"",""意大利王國"",""Regno d'Italia"",""Zjednoczone Królestwo Włoch"",""مملكة إيطاليا"",null},8}, {""http://www.wikidata.org/entity/Q38"",{""Italy"",""Italien"",""Italie"",""Италия"",""意大利"",""Italia"",""Włochy"",""إيطاليا"",""इटली""},9}],2}"
"[{""2016-10-07T00:00:00Z"",{},0}]",1,"{[{""http://www.wikidata.org/entity/Q1489"",{""Mexico City"",""Mexiko-Stadt"",""Mexico"",""Мехико"",""मेक्सिको नगर"",""墨西哥城"",""Città del Messico"",""Meksyk"",""مدينة مكسيكو""},9}],1}","{[{""http://www.wikidata.org/entity/Q617"",{""Padua"",""Padua"",""Padoue"",""Падуя"",""पडुआ"",""帕多瓦"",""Padova"",""Padwa"",""بادوفا""},9}],1}","{[{""1930-05-29T00:00:00Z"",{},0}, {""1932-05-29T00:00:00Z"",{},0}],2}","{[{""http://www.wikidata.org/entity/Q33999"",{""actor"",""Schauspieler"",""acteur ou actrice"",""актёр"",""अभिनेता"",""演員"",""attore"",""aktor"",""ممثل""},9}],1}","{[{""http://www.wikidata.org/entity/Q96"",{""Mexico"",""Mexiko"",""Mexique"",""Мексика"",""墨西哥"",""Messico"",""Meksyk"",""المكسيك"",""मेक्सिको""},9}],1}"
"[{""1148-01-01T00:00:00Z"",{},0}]",1,"{[{""http://www.wikidata.org/entity/Q641"",{""Venice"",""Venedig"",""Venise"",""Венеция"",""वेनिस"",""威尼斯"",""Venezia"",""Wenecja"",""البندقية""},9}],1}","{[{""http://www.wikidata.org/entity/Q641"",{""Venice"",""Venedig"",""Venise"",""Венеция"",""वेनिस"",""威尼斯"",""Venezia"",""Wenecja"",""البندقية""},9}],1}","{[{""1098-01-01T00:00:00Z"",{},0}],1}","{[{""http://www.wikidata.org/entity/Q82955"",{""politician"",""Politiker"",""personnalité politique"",""политик"",""राजनीतिज्ञ"",""政治人物"",""politico"",""polityk"",""سياسي""},9}],1}","{[{""http://www.wikidata.org/entity/Q4948"",{""Republic of Venice"",""Republik Venedig"",""république de Venise"",""Венецианская республика"",""威尼斯共和国"",""Repubblica di Venezia"",""Republika Wenecka"",""جمهورية البندقية"",""वेनिस गणराज्य""},9}],1}"
"[{""1925-11-23T00:00:00Z"",{},0}, {""1925-11-29T00:00:00Z"",{},0}]",2,"{[{""http://www.wikidata.org/entity/Q2634"",{""Naples"",""Neapel"",""Naples"",""Неаполь"",""नापोलि"",""那不勒斯"",""Napoli"",""Neapol"",""نابولي""},9}],1}","{[{""http://www.wikidata.org/entity/Q2634"",{""Naples"",""Neapel"",""Naples"",""Неаполь"",""नापोलि"",""那不勒斯"",""Napoli"",""Neapol"",""نابولي""},9}],1}","{[{""1853-03-13T00:00:00Z"",{},0}],1}","{[{""http://www.wikidata.org/entity/Q28389"",{""screenwriter"",""Drehbuchautor"",""scénariste"",""сценарист"",""पटकथा लेखक"",""編劇"",""sceneggiatore"",""scenarzysta"",""كاتب سيناريو""},9}, {""http://www.wikidata.org/entity/Q33999"",{""actor"",""Schauspieler"",""acteur ou actrice"",""актёр"",""अभिनेता"",""演員"",""attore"",""aktor"",""ممثل""},9}, {""http://www.wikidata.org/entity/Q487596"",{""Dramaturg"",""Dramaturg"",""dramaturge"",""драматург-режиссёр"",null,""剧作者"",""drammaturgo"",""dramaturg"",""درامي""},8}],2}","{[{""http://www.wikidata.org/entity/Q172579"",{""Kingdom of Italy"",""Königreich Italien"",""Royaume d'Italie"",""Королевство Италия"",""意大利王國"",""Regno d'Italia"",""Zjednoczone Królestwo Włoch"",""مملكة إيطاليا"